<a href="https://colab.research.google.com/github/hoshi-3104-com/california-house-price-competition/blob/main/notebooks/03_house_price_kfold_vs_holdout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 03. 評価手法の選定方法に関する検証

* Hold-out法とクロスバリデーション法のどちらが精度が良いかを検証する


In [ ]:
# driveのマウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Kaggle APIを利用できるようにするため、kaggle.jsonからusernameとkeyを設定する
import os
import json
f = open("/content/drive/MyDrive/house_price_competion_hoshino/kaggle.json", 'r') # ディレクトリは必要に応じて変更してください
json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# 前処理(正規化・標準化)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.linear_model import LinearRegression

# 精度評価
from sklearn.metrics import mean_squared_error

# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/train_data_raw.csv')
test = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/test_data_raw.csv')
sample = pd.read_csv('/content/drive/MyDrive/house_price_competion_hoshino/sample_raw.csv')

In [ ]:
train.head()

,id,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Household,AllRooms,AllBedrms,Price
0,0,1.4817,6.0,4.443645,1.134293,1397.0,3.350120,36.77,-119.84,417.0,1853.0,473.0,0.720
1,1,6.9133,8.0,5.976471,1.026471,862.0,2.535294,33.68,-117.80,340.0,2032.0,349.0,2.741
2,2,1.5536,25.0,4.088785,1.000000,931.0,4.350467,36.60,-120.19,214.0,875.0,214.0,0.583
3,3,1.5284,31.0,2.740088,1.008811,597.0,2.629956,34.10,-118.32,227.0,622.0,229.0,2.000
4,4,4.0815,21.0,5.166667,1.002688,1130.0,3.037634,37.79,-121.23,372.0,1922.0,373.0,1.179


In [ ]:
# データを説明変数と目的変数に分ける
train_X = train.drop(['Price', 'id'], axis=1)
target = train['Price']

print('データ分割前の行数' ,len(train_X))

データ分割前の行数 16512


#### **①ホールドアウト法でのモデル構築**

In [ ]:
# train_test_splitのimport
from sklearn.model_selection import train_test_split

# データの分割(学習用データ80% 検証用データ 20%)
# test_sizeで検証用データの割合を設定することができる

X_train, X_test, y_train, y_test = train_test_split(train_X, target, test_size=0.2, random_state=2)

# 分割後のデータ数を確認
print('学習用データの行数' ,len(X_train))
print('検証用用データの行数' ,len(X_test))

学習用データの行数 13209
検証用用データの行数 3303


In [ ]:
# REF法を利用した特徴量選択（変数選択）
from sklearn.feature_selection import RFE

# インスタンス
rfe = RFE(LinearRegression())

# 特徴量選定（変数選択）の実施
X_rfe = rfe.fit(X_train,y_train)
X_selected = X_train.columns[X_rfe.support_]

# 選択した特徴量
print(X_selected)

Index(['MedInc', 'AveRooms', 'AveBedrms', 'Latitude', 'Longitude'], dtype='object')


In [ ]:
# xgboostのimport
from xgboost import XGBRFRegressor
# xgboostのインスタンス化
XGBoost = xgb.XGBRegressor()

# 学習の実行
model = XGBoost.fit(X_train[X_selected], y_train)

# モデルの予測 学習済みモデルを使用して、新しいデータに対して予測を行うために、predict関数を利用できます。
pred_xg_train = model.predict(X_train[X_selected])
pred_xg_test = model.predict(X_test[X_selected])

# 数値の丸め込み処理 0以下を0に5.00001以上を5.00001に変換する
pred_xg_train_rounding = np.where(pred_xg_train <= 0, 0, pred_xg_train)
pred_xg_train_rounding = np.where(pred_xg_train_rounding >= 5.00001, 5.00001, pred_xg_train_rounding)

pred_xg_test_rounding = np.where(pred_xg_test <= 0, 0, pred_xg_test)
pred_xg_test_rounding = np.where(pred_xg_test_rounding >= 5.00001, 5.00001, pred_xg_test_rounding)

In [ ]:
# 学習データ全てを使って学習します
model = XGBoost.fit(train_X[X_selected], target)

# モデルの予測 学習済みモデルを使用して、新しいデータに対して予測を行うために、predict関数を利用できます。
pred_xg = model.predict(test[X_selected])

# 数値の丸め込み処理 0以下を0に5.00001以上を5.00001に変換する
pred_xg_rounding = np.where(pred_xg <= 0, 0, pred_xg)
pred_xg_rounding = np.where(pred_xg_rounding >= 5.00001, 5.00001, pred_xg_rounding)

In [ ]:
# sampleに予測値の代入と提出ファイルの作成
sample['Price'] = pred_xg_rounding
sample

,id,Price
0,0,3.005660
1,1,1.930357
2,2,0.836290
3,3,3.835821
4,4,3.807373
...,...,...
4123,4123,1.228203
4124,4124,0.720095
4125,4125,1.722244
4126,4126,1.052710


In [ ]:
# csvファイルの作成
sample.to_csv('submit_raw_holdout_xgb.csv',index=None)

In [ ]:
# 作成したファイルをKaggleに直接投稿
# -c:提出コンペ名
# -f：ファイル名
# -m：投稿コメント

!kaggle competitions submit -c ambl-california-housing -f submit_raw_holdout_xgb.csv -m "Yeah! I submit my file through the Google Colab!"

100% 58.8k/58.8k [00:00<00:00, 87.3kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing

#### **②クロスバリデーション法でのモデル構築**

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_selection import RFE  # RFE（特徴量削減）をインポート
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

train = pd.read_csv(
    "/content/drive/MyDrive/house_price_competion_hoshino/train_data_raw.csv"
)
test = pd.read_csv(
    "/content/drive/MyDrive/house_price_competion_hoshino/test_data_raw.csv"
)
sample = pd.read_csv(
    "/content/drive/MyDrive/house_price_competion_hoshino/sample_raw.csv"
)

# 2. データの切り分け
X_all_train = train.drop(["Price", "id"], axis=1)
y_all_train = train["Price"]
X_test = test.drop(["id"], axis=1) if "id" in test.columns else test

print("データ分割前の行数:", len(X_all_train))

# 3. クロスバリデーションおよびRFEの設定
folds_num = 5
kf = KFold(n_splits=folds_num, shuffle=True, random_state=42)

# 各種予測値やスコアを格納する配列
oof_predictions = np.zeros(len(X_all_train))
test_predictions = np.zeros(len(X_test))
cv_scores = []

# 4. クロスバリデーションのループ処理（RFEを含む）
for fold, (cv_train_idx, cv_val_idx) in enumerate(
    kf.split(X_all_train, y_all_train)
):
    print(f"--- Fold {fold + 1} / {folds_num} 開始 ---")

    # 1. このフォールドの訓練データと検証データに分割
    X_cv_train, y_cv_train = (
        X_all_train.iloc[cv_train_idx],
        y_all_train.iloc[cv_train_idx],
    )
    X_cv_val, y_cv_val = (
        X_all_train.iloc[cv_val_idx],
        y_all_train.iloc[cv_val_idx],
    )

    # 2. RFE（特徴量選択）の実行
    print(f"Fold {fold + 1}: RFEによる特徴量選択中...")
    rfe_base_model = xgb.XGBRegressor(random_state=42)
    rfe_selector = RFE(
        estimator=rfe_base_model,
        step=1,
    )

    # このフォールドの訓練データのみを使って、どの特徴量を残すか学習
    rfe_selector.fit(X_cv_train, y_cv_train)

    # 3. 選択された「無駄のない特徴量」だけにデータを絞り込む
    X_cv_train_selected = rfe_selector.transform(X_cv_train)
    X_cv_val_selected = rfe_selector.transform(X_cv_val)
    X_test_selected = rfe_selector.transform(X_test)

    selected_features = X_all_train.columns[rfe_selector.support_]
    print(f"選ばれた特徴量: {list(selected_features)}")

    # 4. 絞り込んだ特徴量を使って、最終的な予測モデルを学習
    model = xgb.XGBRegressor(random_state=42)
    model.fit(X_cv_train_selected, y_cv_train)

    # 5. 検証データおよびテストデータに対する予測
    val_preds = model.predict(X_cv_val_selected)

    # 【追加】検証データの数値の丸め込み処理
    val_preds = np.where(val_preds <= 0, 0, val_preds)
    val_preds = np.where(val_preds >= 5.00001, 5.00001, val_preds)

    oof_predictions[cv_val_idx] = val_preds

    # 各フォールドで絞り込まれたテストデータを使って予測し、平均化
    test_preds = model.predict(X_test_selected)

    # 【追加】テストデータの数値の丸め込み処理
    test_preds = np.where(test_preds <= 0, 0, test_preds)
    test_preds = np.where(test_preds >= 5.00001, 5.00001, test_preds)

    test_predictions += test_preds / folds_num

    # 6. このフォールドのスコアを記録
    fold_rmse = np.sqrt(mean_squared_error(y_cv_val, val_preds))
    cv_scores.append(fold_rmse)
    print(f"Fold {fold + 1} の検証スコア (RMSE): {fold_rmse:.4f}")

# 5. クロスバリデーション全体の評価スコア算出
print("\n==================================")
print(f"各Foldの平均スコア: {np.mean(cv_scores):.4f}")
overall_oof_score = np.sqrt(mean_squared_error(y_all_train, oof_predictions))
print(f"全体のOOFスコア (RMSE): {overall_oof_score:.4f}")
print("==================================")

# 6. 提出用ファイルの作成
sample["Price"] = test_predictions

# CSVとして出力（必要に応じてコメントアウトを解除してください）
# sample.to_csv('/content/drive/MyDrive/house_price_competion_hoshino/submission_cv_rfe.csv', index=False)
# print("提出用ファイルを保存しました。")

データ分割前の行数: 16512
--- Fold 1 / 5 開始 ---
Fold 1: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'HouseAge', 'AveOccup', 'Latitude', 'Longitude']
Fold 1 の検証スコア (RMSE): 0.4886
--- Fold 2 / 5 開始 ---
Fold 2: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'HouseAge', 'AveOccup', 'Latitude', 'Longitude']
Fold 2 の検証スコア (RMSE): 0.4708
--- Fold 3 / 5 開始 ---
Fold 3: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'HouseAge', 'AveOccup', 'Latitude', 'Longitude']
Fold 3 の検証スコア (RMSE): 0.4804
--- Fold 4 / 5 開始 ---
Fold 4: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'HouseAge', 'AveOccup', 'Latitude', 'Longitude']
Fold 4 の検証スコア (RMSE): 0.4624
--- Fold 5 / 5 開始 ---
Fold 5: RFEによる特徴量選択中...
選ばれた特徴量: ['MedInc', 'HouseAge', 'AveOccup', 'Latitude', 'Longitude']
Fold 5 の検証スコア (RMSE): 0.4896

各Foldの平均スコア: 0.4784
全体のOOFスコア (RMSE): 0.4785


In [ ]:
# csvファイルの作成
sample.to_csv('submit_raw_cv_xgb.csv',index=None)

In [ ]:
# 作成したファイルをKaggleに直接投稿
# -c:提出コンペ名
# -f：ファイル名
# -m：投稿コメント

!kaggle competitions submit -c ambl-california-housing -f submit_raw_cv_xgb.csv -m "Yeah! I submit my file through the Google Colab!"

100% 94.1k/94.1k [00:00<00:00, 135kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing